# Retail Sales & Customer Analytics Platform
## Notebook 02: Customer RFM Segmentation & Machine Learning Clustering
**Role:** Senior Data Scientist & ML Engineer

This notebook walks through behavioral customer intelligence:
1. Calculation of Recency (R), Frequency (F), and Monetary Value (M).
2. Rule-based RFM scoring and segment categorization (Champions, Loyal Customers, At Risk, Can't Lose Them, etc.).
3. Unsupervised K-Means clustering with feature log-transforms, standardization, and Elbow/Silhouette evaluation.
4. Business interpretation and marketing strategy.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.segmentation import run_full_segmentation_pipeline

%matplotlib inline
sns.set_theme(style='whitegrid')


### 1. Execute Customer RFM Pipeline


In [ ]:
rfm_df = run_full_segmentation_pipeline()
print(f'Successfully segmented {len(rfm_df)} registered customer accounts.')
rfm_df.head()


### 2. Segment Distribution & Revenue Contribution


In [ ]:
seg_summary = rfm_df.groupby('Customer_Segment').agg(
    Customer_Count=('CustomerID', 'count'),
    Total_Revenue=('Monetary', 'sum'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Spend=('Monetary', 'mean')
).reset_index().sort_values('Total_Revenue', ascending=False)

seg_summary['Revenue_Share_%'] = (seg_summary['Total_Revenue'] / rfm_df['Monetary'].sum() * 100).round(2)
display(seg_summary)


### 3. Visualizing Customer Segments: Volume vs Value


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=seg_summary, y='Customer_Segment', x='Customer_Count', palette='viridis', ax=ax1)
ax1.set_title('Customer Count by RFM Segment')
ax1.set_xlabel('Accounts')
ax1.set_ylabel('')

sns.barplot(data=seg_summary, y='Customer_Segment', x='Total_Revenue', palette='plasma', ax=ax2)
ax2.set_title('Total Revenue Contribution (£ GBP)')
ax2.set_xlabel('Revenue (£)')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()


### 4. K-Means Clustering Profiles


In [ ]:
cluster_summary = rfm_df.groupby(['KMeans_Cluster', 'Cluster_Name']).agg(
    Accounts=('CustomerID', 'count'),
    Mean_Recency=('Recency', 'mean'),
    Mean_Frequency=('Frequency', 'mean'),
    Mean_Spend=('Monetary', 'mean'),
    Total_Spend=('Monetary', 'sum')
).reset_index().sort_values('Total_Spend', ascending=False)

display(cluster_summary)
